# Research foundations: what do my saved order books tell me?

**Question:** what visible liquidity did OKX and Bitfinex report, and how confidently
can I reconstruct it from saved inputs?

This notebook reads **OKX BTC-USDT historical L2 books** (aggregate price levels)
and **Bitfinex BTC/USD recorded live R0 books** (visible individual orders).
It uses the existing package readers. It makes **no network requests** and needs
no account or credentials. To obtain inputs first, run
[market-data-capture.ipynb](market-data-capture.ipynb), which records public feeds
and downloads the selected OKX day.

These are order books, not trades. They do not replace the
[frozen P1 trade study](../docs/research/p1-baseline.md). The sources have different
quote currencies, observation times and depth limits; their prices are not a
simultaneous arbitrage comparison. Restart the kernel and Run All in order.

Section 9 adds a separate synthetic trade-origin exercise (NB 09). It needs no
Binance access and leaves the saved order-book exploration intact.


## 1. Which implementation am I using?

The checks below require the pinned interpreter and an installed package whose
runtime files match this checkout. The table identifies this execution so that a
later result can be compared with the same code and saved inputs.


In [1]:
# standard library imports
import hashlib
import json
import os
import platform
import subprocess
import sys
from datetime import UTC, datetime, timedelta
from decimal import Decimal
from html import escape
from importlib.metadata import distributions
from pathlib import Path

# third party library imports
from IPython.display import HTML, display

# project library imports
import alpha_research
from alpha_research.data.live.config import CaptureConfig, CaptureRequest
from alpha_research.data.live.journal import CaptureJournal, CaptureRecord
from alpha_research.data.live.replay import CaptureReplay
from alpha_research.data.raw_objects import RawObject
from alpha_research.data.venues.bitfinex.book import BitfinexBook
from alpha_research.data.venues.okx.history_config import OKXArchiveRequest, OKXHistoryConfig
from alpha_research.data.venues.okx.history_store import OKXArchiveStore
from alpha_research.serialization import DomainCodec

root = Path.cwd().resolve()
if root.name == "examples":
    root = root.parent
assert (root / "configs/history/okx-btc-usdt.json").is_file(), (
    "Run from the repository root or examples/."
)
notebook_path = Path(
    os.environ.get("ALPHA_RESEARCH_NOTEBOOK_PATH", root / "examples/research-foundations.ipynb")
).resolve()


class NotebookDisplay:
    """Render small tables for the exploratory notebook scenarios."""

    @staticmethod
    def show_table(headers: tuple[str, ...], rows: list[tuple[object, ...]]) -> None:
        """Display a small escaped table without adding a dataframe dependency.

        Args:
            headers: Column labels.
            rows: Exact or presentation values to render as text.
        """
        heading = "".join(
            f'<th style="text-align:left;padding:6px">{escape(label)}</th>' for label in headers
        )
        body = "".join(
            "<tr>"
            + "".join(
                f'<td style="padding:6px;vertical-align:top">{escape(str(value))}</td>'
                for value in row
            )
            + "</tr>"
            for row in rows
        )
        display(HTML(f"<table><thead><tr>{heading}</tr></thead><tbody>{body}</tbody></table>"))

In [2]:
assert platform.python_version() == (root / ".python-version").read_text().strip() == "3.14.7"
installed = Path(alpha_research.__file__).resolve().parent
assert "site-packages" in installed.parts and installed != root / "src/alpha_research"
source_root = root / "src/alpha_research"
runtime_files = sorted(
    path
    for path in source_root.rglob("*")
    if path.is_file() and path.suffix in {".py", ".json", ".typed"}
)
assert runtime_files
runtime_digest = hashlib.sha256()
for source_file in runtime_files:
    relative = source_file.relative_to(source_root)
    source_bytes = source_file.read_bytes()
    assert source_bytes == (installed / relative).read_bytes(), f"Stale installed file: {relative}"
    runtime_digest.update(str(relative).encode() + b"\0" + source_bytes + b"\0")

commit = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=root, text=True).strip()
branch = subprocess.check_output(["git", "branch", "--show-current"], cwd=root, text=True).strip()
working_status = subprocess.check_output(["git", "status", "--short"], cwd=root, text=True).strip()
candidate_diff = subprocess.check_output(
    [
        "git",
        "diff",
        "--binary",
        "HEAD",
        "--",
        "src",
        "configs",
        "pyproject.toml",
        "uv.lock",
        ".python-version",
    ],
    cwd=root,
)
notebook_document = json.loads(notebook_path.read_text())
notebook_source = [
    (cell["cell_type"], "".join(cell["source"])) for cell in notebook_document["cells"]
]
notebook_digest = hashlib.sha256(
    json.dumps(notebook_source, ensure_ascii=False, separators=(",", ":")).encode()
).hexdigest()
identity = {
    "notebook_path": str(notebook_path),
    "attempt_id": os.environ.get("ALPHA_RESEARCH_ATTEMPT_ID", "interactive: unsnapshotted"),
    "commit": commit,
    "branch": branch,
    "working_tree": "changed" if working_status else "clean",
    "python": platform.python_version(),
    "platform": platform.platform(),
    "kernel": sys.executable,
    "package": str(installed),
    "verified_runtime_files": len(runtime_files),
    "runtime_sha256": runtime_digest.hexdigest(),
    "runtime_input_diff_sha256": hashlib.sha256(candidate_diff).hexdigest(),
    "uv_lock_sha256": hashlib.sha256((root / "uv.lock").read_bytes()).hexdigest(),
    "okx_config_sha256": hashlib.sha256(
        (root / "configs/history/okx-btc-usdt.json").read_bytes()
    ).hexdigest(),
    "notebook_source_sha256": notebook_digest,
    "notebook_tooling": {
        item.metadata["Name"]: item.version
        for item in distributions()
        if item.metadata["Name"] in {"nbclient", "nbformat", "ipykernel"}
    },
}
NotebookDisplay.show_table(("Identity", "Observed value"), list(identity.items()))

Identity,Observed value
notebook_path,/Users/lxg/Documents/GitHub/Alpha Research/alpha-research-lab-private/examples/research-foundations.ipynb
attempt_id,7ae4500f-760b-4169-8da9-9a3b819eaf11
commit,0969f016e480bc630f7fa0d3d811cc1e1580dc12
branch,codex/mr-09-raw-occurrences
working_tree,changed
python,3.14.7
platform,macOS-27.0-arm64-arm-64bit-Mach-O
kernel,/Users/lxg/.cache/uv/builds-v0/.tmp1rv8dD/bin/python
package,/Users/lxg/Documents/GitHub/Alpha Research/alpha-research-lab-private/.venv/lib/python3.14/site-packages/alpha_research
verified_runtime_files,61


## 2. Which saved samples will I inspect?

Set `ALPHA_OKX_ARCHIVE_DIRECTORY` and `ALPHA_BITFINEX_CAPTURE_DIRECTORY` to the
directories printed by the capture notebook to repeat a particular observation.
Without overrides, this exploration selects the **most recently saved completed
input with the exact maintained configuration**, by manifest save time. That
convenience selection can change after a new acquisition. Pin the printed paths
when a result matters; this is not an accepted research dataset.

The selection table records the paths, fingerprints and original implementation.
The next cells verify the bytes. A selected invalid input fails visibly; it never
triggers a replacement download or fallback to an older sample. No matching input
means run the capture notebook first.


In [3]:
codec = DomainCodec()
okx_config = OKXHistoryConfig.read(root / "configs/history/okx-btc-usdt.json")
bitfinex_config = CaptureConfig.read(root / "configs/live/bitfinex-btc-usd.json")
selected = {}
selection_rows = []
for venue, folder, variable, request_type, config in (
    ("okx", "data/okx/history", "ALPHA_OKX_ARCHIVE_DIRECTORY", OKXArchiveRequest, okx_config),
    (
        "bitfinex",
        "data/bitfinex",
        "ALPHA_BITFINEX_CAPTURE_DIRECTORY",
        CaptureRequest,
        bitfinex_config,
    ),
):
    override = os.environ.get(variable)
    candidates = []
    paths = (
        [Path(override).expanduser()]
        if override
        else [p.parent for p in (root / folder).glob("*/manifest.json")]
    )
    for path in paths:
        request = codec.decode_json((path / "request.json").read_bytes(), request_type)
        if request.config != config:
            if override:
                raise ValueError(f"{variable} differs from the maintained selection")
            continue
        descriptor = codec.decode_json((path / "manifest.json").read_bytes(), RawObject)
        candidates.append((descriptor.ts_ingested, str(path.resolve()), request, descriptor))
    if not candidates:
        raise FileNotFoundError(f"No matching {venue} input; run market-data-capture.ipynb first")
    _, location, request, descriptor = max(candidates, key=lambda item: (item[0], item[1]))
    selected[venue] = (Path(location), request, descriptor)
    selection_rows.extend(
        [
            (
                venue,
                "Selection",
                "explicit directory" if override else "latest matching saved input",
            ),
            (venue, "Directory", location),
            (venue, "Raw SHA-256", descriptor.sha256),
            (venue, "Request identity", request.identity),
            (venue, "Original implementation SHA-256", request.implementation_identity),
        ]
    )
NotebookDisplay.show_table(("Source", "Evidence", "Value"), selection_rows)

Source,Evidence,Value
okx,Selection,explicit directory
okx,Directory,/Users/lxg/Documents/GitHub/Alpha Research/alpha-research-lab-private/data/okx/history/802967bb-05eb-4b7b-9c90-2c964f22e41f
okx,Raw SHA-256,fef59b343d1972c32f67583edc538c5d2f7350e6f3a0059e6f87896d7e8bf676
okx,Request identity,71672efd00a8f3f6c959d0a08f8ce2a8793b3fb84d23f1a3c22b2756a6ec2665
okx,Original implementation SHA-256,22c0e38d88691bd0b1141588260465c471bbde0ff432a6bd1d0dab0505000014
bitfinex,Selection,explicit directory
bitfinex,Directory,/Users/lxg/Documents/GitHub/Alpha Research/alpha-research-lab-private/data/bitfinex/a5116c5a-b7d6-4e2b-9ee6-c79839c80f43
bitfinex,Raw SHA-256,a57ddabc5b1283c702ee2c7fd104a2b9cd943e2b41661cfc1d66619b35930163
bitfinex,Request identity,0e1453a2e7e925ad5bc1454a6a7d1e7298eca46e82a0d07b10e7ad8fc1c9ec21
bitfinex,Original implementation SHA-256,22c0e38d88691bd0b1141588260465c471bbde0ff432a6bd1d0dab0505000014


## 3. How much of the historical OKX day did I reconstruct?

The configured date is **8 September 2026**. The reader verifies the original
archive, then reconstructs at most **10,000 records**. The displayed source-time
interval is the processed prefix, not the whole day. L2 combines all reported
size at each price; it supplies no individual order identity or queue position.


In [4]:
okx_directory, okx_request, okx_descriptor = selected["okx"]
history = OKXArchiveStore(okx_directory).replay()
assert history["source_sequence"] is None and history["gaps"] is None
epoch = datetime(1970, 1, 1, tzinfo=UTC)
history_rows = [
    (key, history[key])
    for key in (
        "symbol",
        "representation",
        "records",
        "snapshots",
        "updates",
        "levels",
        "depth",
        "state_available",
        "continuity",
        "stop_reason",
        "entire_archive_replayed",
    )
]
for key in ("first_source_ms", "last_source_ms"):
    value = history[key]
    history_rows.append(
        (
            key.replace("_ms", "_utc"),
            None if value is None else epoch + timedelta(milliseconds=int(value)),
        )
    )
NotebookDisplay.show_table(("Historical observation", "Result"), history_rows)

Historical observation,Result
symbol,BTC-USDT
representation,L2
records,10000
snapshots,1
updates,9999
levels,800
depth,400
state_available,True
continuity,unproven_no_source_sequences
stop_reason,record_limit


**Read the result:** `gaps=None` means continuity is unknown: this archive has no
source sequence numbers. `state_available=True` means a book could be built from
the observed prefix; it does not certify uninterrupted historical coverage.
The archive API currently returns a reconstruction summary, not a research table
of historical prices or features.


## 4. What did the Bitfinex recording actually observe?

This is a saved bounded live recording, not downloadable Bitfinex historical data.
The table reports local receipt times and observed gaps. The R0 subscription
requests depth 25; the final number of retained visible orders is reported
separately. Sequence continuity covers this recording only.


In [5]:
bitfinex_directory, bitfinex_request, bitfinex_descriptor = selected["bitfinex"]
replay = CaptureReplay.read(bitfinex_directory)
summary = replay.summary()
assert isinstance(replay.book, BitfinexBook)
live_rows = [
    (key, summary[key])
    for key in (
        "symbol",
        "messages",
        "snapshots",
        "updates",
        "orders",
        "depth",
        "gaps",
        "synchronized",
        "source_sequence",
        "last_source_time",
    )
]
for key in ("started_ns", "finished_ns"):
    live_rows.append(
        (key.replace("_ns", "_utc"), epoch + timedelta(microseconds=summary[key] // 1000))
    )
NotebookDisplay.show_table(("Recorded observation", "Result"), live_rows)
assert summary["synchronized"] and not summary["gaps"], (
    "This sample is unsuitable for the uninterrupted-book preview; inspect its gaps"
)

Recorded observation,Result
symbol,tBTCUSD
messages,376
snapshots,1
updates,371
orders,50
depth,25
gaps,[]
synchronized,True
source_sequence,373
last_source_time,None


## 5. What visible prices and sizes can I inspect?

Read a few individual Bitfinex orders nearest each side's best price. Prices are
**USD per BTC**, sizes are **BTC**, and the displayed spread is **USD per BTC**.
Decimal values remain exact and are rendered as text. The spread is a descriptive
observation, not a trading return; it excludes fees, impact and fill uncertainty.


In [6]:
preview_rows = []
best = {}
for side in ("buy", "sell"):
    orders = sorted(
        (order for order in replay.book.orders.values() if order.side == side),
        key=lambda order: order.price,
        reverse=side == "buy",
    )
    assert orders, f"No observed {side} side"
    best[side] = orders[0].price
    for order in orders[:3]:
        assert isinstance(order.price, Decimal) and isinstance(order.size, Decimal)
        preview_rows.append((side, order.order_id, str(order.price), str(order.size)))
NotebookDisplay.show_table(("Side", "Visible order ID", "USD / BTC", "BTC"), preview_rows)
NotebookDisplay.show_table(
    ("Measure", "USD / BTC"),
    [
        ("Best bid", best["buy"]),
        ("Best ask", best["sell"]),
        ("Observed spread", best["sell"] - best["buy"]),
    ],
)

Side,Visible order ID,USD / BTC,BTC
buy,244232632883,75645,0.00180442
buy,244230114777,75639,0.0002
buy,244218842321,75638,0.0002
sell,244216587539,75655,0.00022
sell,244232116576,75660,0.03522185
sell,244217678512,75660,0.00022


Measure,USD / BTC
Best bid,75645
Best ask,75655
Observed spread,10


## 6. When did the market event happen, and when did I know about it?

OKX historical records have source timestamps, but their download time is much
later and there is no historical receipt clock. Bitfinex R0 has local receipt order
and sequences, but no exchange timestamp in these book messages. Neither clock
can fill in the other's missing information. UTC display truncates nanoseconds to
microseconds; original Bitfinex nanoseconds remain in the journal.


In [7]:
NotebookDisplay.show_table(
    ("Sample", "Source clock", "Local receipt evidence", "Saved bytes"),
    [
        (
            "OKX historical BTC-USDT",
            "Unix milliseconds in records; continuity unknown",
            f"Archive received {history['download_received']}; per-record receipt absent",
            history["saved_bytes"],
        ),
        (
            "Bitfinex recorded BTC/USD",
            "Absent in R0 messages",
            f"Journal nanoseconds {summary['started_ns']} to {summary['finished_ns']}",
            bitfinex_descriptor.size_bytes,
        ),
    ],
)

Sample,Source clock,Local receipt evidence,Saved bytes
OKX historical BTC-USDT,Unix milliseconds in records; continuity unknown,Archive received 2026-09-16T08:49:14.475436+00:00; per-record receipt absent,109423187
Bitfinex recorded BTC/USD,Absent in R0 messages,Journal nanoseconds 1789548508888411000 to 1789548529076738000,52908


## 7. What prevents me from analysing damaged or disconnected input?

These **controlled failures** modify temporary copies only: remove saved input,
change a saved byte, alter archive discovery, and inject a disconnect into a
separate reconstruction. The original recordings stay intact. Inspect the explicit
rejections and the unavailable book after the gap. A fresh subscription/snapshot
is needed for recovery; this example does not fill the gap with stale quotes.


In [8]:
# standard library imports
import shutil
from tempfile import TemporaryDirectory

failure_rows = []
with TemporaryDirectory(prefix="orderbook-input-checks-") as temporary:
    copy = Path(temporary)
    for name in ("request.json", "manifest.json"):
        shutil.copyfile(bitfinex_directory / name, copy / name)
    raw = copy / bitfinex_descriptor.sha256
    for label, expected in (
        ("Missing Bitfinex bytes", FileNotFoundError),
        ("Changed Bitfinex bytes", ValueError),
    ):
        if expected is ValueError:
            content = (bitfinex_directory / bitfinex_descriptor.sha256).read_bytes()
            raw.write_bytes(bytes([content[0] ^ 1]) + content[1:])
        try:
            CaptureReplay.read(copy)
        except expected as error:
            failure_rows.append((label, type(error).__name__))
        else:
            raise AssertionError(f"Accepted {label}")
    shutil.copyfile(okx_directory / "request.json", copy / "request.json")
    (copy / "index.json").write_bytes(b"{}")
    try:
        OKXArchiveStore(copy).replay()
    except ValueError as error:
        failure_rows.append(("Changed OKX discovery", str(error)))
    else:
        raise AssertionError("Accepted changed OKX discovery")

exercise = CaptureReplay(bitfinex_config)
for record in CaptureJournal(bitfinex_directory, bitfinex_request).records():
    if record.kind == "end":
        break
    exercise.apply(record)
exercise.apply(
    CaptureRecord(
        kind="gap",
        epoch=exercise.epoch,
        received_ns=exercise.finished_ns,
        reason="synthetic_disconnect",
    )
)
assert not exercise.book.ready
failure_rows.append(
    ("Injected disconnect", f"Book usable: {exercise.book.ready}; {exercise.gaps[-1]['reason']}")
)
NotebookDisplay.show_table(("Controlled case", "Observed result"), failure_rows)

Controlled case,Observed result
Missing Bitfinex bytes,FileNotFoundError
Changed Bitfinex bytes,ValueError
Changed OKX discovery,saved OKX archive discovery differs from request
Injected disconnect,Book usable: False; synthetic_disconnect


## 8. What can I conclude, and what should I investigate next?

The last check reopens the Bitfinex input and compares the entire sequence of
reconstructed states. This establishes repeatability from the same saved bytes
and installed code, not data completeness or executable fills.


In [9]:
again = CaptureReplay.read(bitfinex_directory)
assert again.summary()["trace_identity"] == summary["trace_identity"]
NotebookDisplay.show_table(
    ("Research finding", "Evidence or limit"),
    [
        (
            "Historical liquidity observations",
            f"OKX: {history['records']} prefix records, {history['levels']} final levels",
        ),
        (
            "Recorded visible orders",
            f"Bitfinex: {summary['messages']} messages, {summary['orders']} final orders",
        ),
        ("Repeatability", f"Matching Bitfinex state trace: {summary['trace_identity']}"),
        ("Controlled failures", f"{len(failure_rows)} explicit outcomes"),
        (
            "Not established",
            "Same-time prices, full coverage, historical eligibility, queue priority or fills",
        ),
        (
            "Next roadmap MR",
            "MR 10: admit economic duplicates and corrections explicitly; still planned",
        ),
    ],
)

Research finding,Evidence or limit
Historical liquidity observations,"OKX: 10000 prefix records, 800 final levels"
Recorded visible orders,"Bitfinex: 376 messages, 50 final orders"
Repeatability,Matching Bitfinex state trace: 283533b937763c40d108517fc22ddbbc2d10a77820a8de498cc631f37c11d375
Controlled failures,4 explicit outcomes
Not established,"Same-time prices, full coverage, historical eligibility, queue priority or fills"
Next roadmap MR,MR 10: admit economic duplicates and corrections explicitly; still planned


For a new observation, deliberately run the [capture notebook](market-data-capture.ipynb)
and pin its printed directories before rerunning this study. For timing, exact
financial units and saved-row boundary exercises, use the independent
[offline reliability lab](foundation-contract-checks.ipynb). Keep an influencing
trial in the research journal; these local snapshots are not its backup.


## 9. Can two observations have sequence zero without being the same occurrence? (NB 09)

**Sample and assumptions:** the following tiny Binance individual-trade archives
are synthetic teaching inputs, not new market observations. Two versions of a BTC
file start with the same trade; another file belongs to ETH. We resolve the existing
configuration, publish exact raw bytes locally, then use the installed dataset
pipeline to decode them. No network, revision policy or dataset publication is involved.

Read the table from the input binding and raw digest to source position, then to
occurrence identity. A repeated build must reproduce these identities. Trade IDs
are economic identifiers with separate duplicate/correction rules planned in MR 10.

In [10]:
# standard library imports
from zipfile import ZIP_DEFLATED, ZipFile, ZipInfo

# third party library imports
from msgspec.structs import replace

# project library imports
from alpha_research.configuration import Configuration
from alpha_research.data.events import Disconnect
from alpha_research.data.manifests import AcquisitionManifest
from alpha_research.data.normalization import Occurrence
from alpha_research.data.providers.binance.trades import ArchiveTrade
from alpha_research.data.providers.tardis.decoder import TardisDecoder
from alpha_research.data.raw_store import RawStore
from alpha_research.data.requests import SourceRequest
from alpha_research.data.venues.binance.payloads import BinancePayloadDecoder
from alpha_research.pipelines.dataset import iter_occurrences
from alpha_research.reference_data import CatalogSnapshot

archive_config = Configuration.read(root / "configs/data-jobs/binance-archive.json")
archive_catalog = codec.decode_json(
    (root / "configs/reference/binance-archive-spot.json").read_bytes(), CatalogSnapshot
)
archive_run = archive_config.resolve("btc-archive", archive_catalog)
archive_run = replace(archive_run, instruments=archive_run.catalog.instruments)
archive_requests = SourceRequest.partition(archive_run)
mr09_first = b"12,100.00,2.0,199.999,1717200000001,true,true\r\n"
mr09_second = b"11,101.00,3.0,303.000,1717200000001,false,true\n"
mr09_items = []
mr09_failures = []
with TemporaryDirectory(prefix="mr09-occurrences-") as temporary:
    local = Path(temporary)
    store = RawStore(local)
    descriptors = []
    for binding, content in zip(
        archive_requests, (mr09_first + mr09_second, mr09_first), strict=True
    ):
        path = local / "source.zip"
        member = ZipInfo(f"{binding.instrument.symbol}-trades-{binding.start.date()}.csv")
        member.compress_type = ZIP_DEFLATED
        with ZipFile(path, "w") as archive:
            archive.writestr(member, content)
        descriptor = RawObject.from_file(
            path,
            request=binding,
            retrieval_reference="synthetic:nb09",
            ts_received=datetime(2026, 9, 16, tzinfo=UTC),
            ts_ingested=datetime(2026, 9, 16, tzinfo=UTC),
        )
        store.publish(path, descriptor=descriptor, request=binding)
        descriptors.append(descriptor)
    state = AcquisitionManifest(run=archive_run, completed=tuple(descriptors))
    state.save(local / "manifest.json")
    restored = AcquisitionManifest.read(local / "manifest.json", run=archive_run, store=store)
    mr09_items = list(iter_occurrences(restored, store, max_bytes=4096))
    assert list(iter_occurrences(restored, store, max_bytes=4096)) == mr09_items
    assert [item.position for item in mr09_items] == [0, 1, 0]
    assert len({item.identity for item in mr09_items}) == 3
    # An additional file version starts with the same BTC trade and sequence zero.
    with ZipFile(path, "w", compression=ZIP_DEFLATED) as archive:
        archive.writestr("BTCUSDT-trades-2024-06-01.csv", mr09_first)
    original_run = replace(archive_run, instruments=(archive_run.instruments[0],))
    original_request = SourceRequest.partition(original_run)[0]
    original = RawObject.from_file(
        path,
        request=original_request,
        retrieval_reference="synthetic:nb09-original",
        ts_received=descriptors[0].ts_received,
        ts_ingested=descriptors[0].ts_ingested,
    )
    store.publish(path, descriptor=original, request=original_request)
    previous = next(
        iter_occurrences(
            AcquisitionManifest(run=original_run, completed=(original,)), store, max_bytes=4096
        )
    )
    assert previous.position == mr09_items[0].position == 0
    assert previous.event == mr09_items[0].event
    assert previous.identity != mr09_items[0].identity
    mr09_items.append(previous)
    try:
        replace(previous, event=mr09_items[2].event)
    except ValueError as error:
        mr09_failures.append(("Unselected instrument", str(error)))
    else:
        raise AssertionError("foreign event accepted")
    try:
        replace(previous, request=archive_requests[1])
    except ValueError as error:
        mr09_failures.append(("Incompatible raw binding", str(error)))
    else:
        raise AssertionError("foreign raw binding accepted")

NotebookDisplay.show_table(
    ("Binding", "Raw digest", "Position", "Trade ID", "Occurrence identity"),
    [
        (
            item.request.instrument.symbol,
            item.origin.sha256,
            item.position,
            item.event.trade_id,
            item.identity,
        )
        for item in mr09_items
    ],
)

Binding,Raw digest,Position,Trade ID,Occurrence identity
BTCUSDT,f2afa9ba9084181772ee567881d4a3fbad26b5a658fee580a374c9cb3fe5c600,0,12,55816695278b7b11512b3df16b04530d34323935e6f4784351dce793a66e8982
BTCUSDT,f2afa9ba9084181772ee567881d4a3fbad26b5a658fee580a374c9cb3fe5c600,1,11,cd2c559883a17cfa5bd97aecc259456dd2d7fbcbc1700faa4b499b26a9ba5782
ETHUSDT,633cbfce42192b841fdc133b3c865165c5b1f958928287f07b315747561dba7d,0,12,2fe1176c1fd8755bf7d3ddd2995f6ccac709c61d296fead1c33567bc97c4eb6d
BTCUSDT,f99c6b397a037b601c593560b51a589434a4df45e84c55979f3580e941f1d3ce,0,12,c0c3cf9f1f9beeb0656e97d32933fa711ca1d60c813f3ddd6f956ab3e7c47796


**Observed output:** all four occurrences remain attributable, including both
positions zero in the BTC file versions and the independent ETH binding. The two
rows at equal exchange time retain their file order (trade IDs 12 then 11).

The next table retains the publisher's integer milliseconds and decimal scales.
The synthetic quote quantity deliberately differs from price × quantity; decoding
preserves it for later validation. Download receipt belongs to the raw object.
Capture and historical knowledge are absent. No Tardis timestamp is fabricated.

In [11]:
archive_event = mr09_items[0].event
assert isinstance(archive_event, ArchiveTrade)
assert archive_event.ts_source_received is archive_event.ts_knowledge is None
assert archive_event.quote_quantity == Decimal("199.999")
assert codec.decode_json(codec.encode_json(mr09_items[0]), Occurrence) == mr09_items[0]
NotebookDisplay.show_table(
    ("Field", "Value"),
    [
        ("Source integer / unit", (archive_event.timestamp, archive_event.timestamp_unit)),
        ("UTC event time", archive_event.ts_event),
        (
            "Price / base quantity / quote quantity",
            (
                str(archive_event.price),
                str(archive_event.quantity),
                str(archive_event.quote_quantity),
            ),
        ),
        (
            "Capture / historical knowledge",
            (archive_event.ts_source_received, archive_event.ts_knowledge),
        ),
        ("Raw acquisition receipt", mr09_items[0].origin.ts_received),
        ("Original CSV including terminator", repr(mr09_items[0].raw)),
    ],
)

Field,Value
Source integer / unit,"(1717200000001, 'ms')"
UTC event time,2024-06-01 00:00:00.001000+00:00
Price / base quantity / quote quantity,"('100.00', '2.0', '199.999')"
Capture / historical knowledge,"(None, None)"
Raw acquisition receipt,2026-09-16 00:00:00+00:00
Original CSV including terminator,"b'12,100.00,2.0,199.999,1717200000001,true,true\r\n'"


### Which input does an instrument-less disconnect affect?

Use the unchanged Tardis decoder on identical synthetic blank files for two
independent bindings. Both controls say `instrument_id=None` and `stream=global`,
but their occurrence envelopes retain BTC and ETH respectively. Their raw digests
and legacy event IDs can match; their occurrence identities must differ. These
saved-file subscriptions have no live session identifier. Later query/replay work
must retain the binding rather than applying a control to every repository input.

In [12]:
fixture_config = Configuration.read(root / "configs/data-jobs/fixture-inspection.json")
fixture_catalog = codec.decode_json(
    (root / "configs/reference/binance-spot.json").read_bytes(), CatalogSnapshot
)
captured_run = fixture_config.resolve("both-fixture", fixture_catalog)
controls = []
with TemporaryDirectory(prefix="mr09-controls-") as temporary:
    path = Path(temporary) / "blank.txt"
    path.write_bytes(b"\r\n")
    for binding in SourceRequest.partition(captured_run)[:2]:
        origin = RawObject.from_file(
            path,
            request=binding,
            retrieval_reference="synthetic:nb09-control",
            ts_received=datetime(2026, 9, 16, tzinfo=UTC),
            ts_ingested=datetime(2026, 9, 16, tzinfo=UTC),
        )
        control = TardisDecoder(
            BinancePayloadDecoder(captured_run.instruments), binding.instrument.source
        ).decode_line(
            "",
            source_sequence=0,
            local_received=origin.ts_received,
            local_ingested=origin.ts_ingested,
            source_digest=origin.sha256,
            disconnect_capture_time=binding.end,
            disconnect_capture_reason="interval_end",
        )
        assert isinstance(control, Disconnect)
        controls.append(
            Occurrence(
                request=binding, origin=origin, position=0, raw=path.read_bytes(), event=control
            )
        )
assert controls[0].origin.sha256 == controls[1].origin.sha256
assert controls[0].event == controls[1].event
assert controls[0].identity != controls[1].identity
NotebookDisplay.show_table(
    ("Affected binding", "Instrument / stream on control", "Raw digest", "Occurrence identity"),
    [
        (
            item.request.instrument.symbol,
            (item.event.instrument_id, item.event.source_stream),
            item.origin.sha256,
            item.identity,
        )
        for item in controls
    ],
)
NotebookDisplay.show_table(("Required failure", "Observed rejection"), mr09_failures)

Affected binding,Instrument / stream on control,Raw digest,Occurrence identity
BTCUSDT,"(None, 'global')",7eb70257593da06f682a3ddda54a9d260d4fc514f645237f5ca74b08f8da61a6,950f9311d83336c75d97e9cd4bc44cf65192c52d4ba864c2c3504223e5eb1032
ETHUSDT,"(None, 'global')",7eb70257593da06f682a3ddda54a9d260d4fc514f645237f5ca74b08f8da61a6,e8593aab1d041fddfad8869f1bc6121b52ee7dce4bc23b2991d137ad3e03dcee


Required failure,Observed rejection
Unselected instrument,occurrence event instrument differs from selected binding
Incompatible raw binding,occurrence raw origin differs from request binding


**Interpretation and next question:** the installed pipeline can trace decoded
archive trades to exact saved bytes, preserve order and rebuild stable occurrence
identities. The controlled failures reject unselected instruments and incompatible
raw bindings. The retained tests additionally reject malformed/nonfinite amounts,
wrong timestamp units/days, missing or mutated objects and unsafe records.

This is MR 09 occurrence normalization. It does not merge duplicates, order
corrections, establish historical knowledge/eligibility/coverage, or publish an
accepted dataset. The earlier real book summaries remain separate observations.
**Next is MR 10:** determine which occurrences represent the same economic trade
and how correction evidence preserves earlier versions. That work is not started.